In [1]:
# CELL 1: Kết nối dataset Taobao trên Kaggle

from pathlib import Path
import pandas as pd

DATASET_OWNER = "huy2008"
DATASET_SLUG = "taobao-00"
DATASET_FILE = "UserBehavior.csv"

INPUT_DIR = Path("/kaggle/input")

print("=== KIỂM TRA MÔI TRƯỜNG ===")

if not INPUT_DIR.exists():
    raise RuntimeError(
        "Notebook chưa chạy trên Kaggle. "
        "Hãy mở Notebook trên Kaggle và thêm dataset."
    )

# Tìm file CSV trong các thư mục Kaggle Input
csv_files = [
    path
    for path in INPUT_DIR.rglob(DATASET_FILE)
    if DATASET_SLUG in path.parts
    and DATASET_OWNER in path.parts
]

if not csv_files:
    raise FileNotFoundError(
        "Không tìm thấy dataset huy2008/taobao-00.\n"
        "Trên Kaggle, chọn Add Input và thêm dataset TaoBao_00."
    )

CSV_PATH = csv_files[0]

print("Môi trường: Kaggle")
print("Dataset:", f"{DATASET_OWNER}/{DATASET_SLUG}")
print("Đường dẫn CSV:", CSV_PATH)
print("Dung lượng:", round(CSV_PATH.stat().st_size / 1024**3, 2), "GB")
print("Trạng thái: Sẵn sàng xử lý")

=== KIỂM TRA MÔI TRƯỜNG ===


RuntimeError: Notebook chưa chạy trên Kaggle. Hãy mở Notebook trên Kaggle và thêm dataset.

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

print("Kaggle input tồn tại:", INPUT_DIR.exists())

if INPUT_DIR.exists():
    print("\nDanh sách thư mục và file:")

    for path in INPUT_DIR.iterdir():
        print(path)

        if path.is_dir():
            for file in path.rglob("*"):
                if file.is_file():
                    print("  File:", file)
else:
    print("Không tìm thấy /kaggle/input")

In [ ]:
# CELL 3: Xử lý 100% dữ liệu Taobao
# Không lấy mẫu, không tải CSV về máy cá nhân

import time

COLUMNS = [
    "user_id",
    "item_id",
    "category_id",
    "behavior",
    "timestamp"
]

CHUNK_SIZE = 500_000
PRINT_INTERVAL = 5_000_000

total_rows = 0

behavior_counts = {
    "pv": 0,
    "fav": 0,
    "cart": 0,
    "buy": 0
}

start_time = time.perf_counter()

print("=== BẮT ĐẦU XỬ LÝ TAOBAO 100% ===")
print("Dataset:", CSV_PATH.name)
print("Chunk size:", f"{CHUNK_SIZE:,}")

for chunk in pd.read_csv(
    CSV_PATH,
    header=None,
    names=COLUMNS,
    chunksize=CHUNK_SIZE,
    dtype={
        "user_id": "int64",
        "item_id": "int64",
        "category_id": "int64",
        "behavior": "category",
        "timestamp": "int64"
    }
):
    total_rows += len(chunk)

    counts = chunk["behavior"].value_counts()

    for behavior, count in counts.items():
        behavior_counts[str(behavior)] = (
            behavior_counts.get(str(behavior), 0)
            + int(count)
        )

    if total_rows % PRINT_INTERVAL == 0:
        print(f"Đã xử lý: {total_rows:,} dòng")

elapsed = time.perf_counter() - start_time

print("\n=== KẾT QUẢ ===")
print(f"Tổng số tương tác: {total_rows:,}")

for behavior in ["pv", "fav", "cart", "buy"]:
    count = behavior_counts[behavior]
    percentage = count / total_rows * 100 if total_rows else 0

    print(f"{behavior}: {count:,} ({percentage:.2f}%)")

print(f"\nThời gian xử lý: {elapsed:.2f} giây")

assert sum(behavior_counts.values()) == total_rows, (
    "Có hành vi ngoài 4 loại pv/fav/cart/buy."
)

print("Kiểm tra dữ liệu: Thành công")

In [ ]:
# ============================================================
# CELL 4 — TAOBAO DATA CLEANING & NORMALIZATION
# Full dataset: 100,150,807 interactions
# Environment: Kaggle Notebook
# ============================================================

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from pathlib import Path
import json
import time
import gc

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

CSV_PATH = Path(CSV_PATH)

OUTPUT_DIR = Path("/kaggle/working/taobao_clean")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 500_000

COLUMNS = [
    "user_id",
    "item_id",
    "category_id",
    "behavior",
    "timestamp"
]

VALID_BEHAVIORS = {"pv", "fav", "cart", "buy"}

BEHAVIOR_ENCODING = {
    "pv": 0,
    "fav": 1,
    "cart": 2,
    "buy": 3
}

# 25/11/2017 00:00:00 UTC+8
START_TS = int(
    pd.Timestamp(
        "2017-11-25 00:00:00",
        tz="Asia/Shanghai"
    ).timestamp()
)

# 04/12/2017 00:00:00 UTC+8 (exclusive)
END_TS = int(
    pd.Timestamp(
        "2017-12-04 00:00:00",
        tz="Asia/Shanghai"
    ).timestamp()
)

print("=" * 60)
print("TAOBAO — DATA CLEANING")
print("=" * 60)

print("Input:", CSV_PATH)
print("Output:", OUTPUT_DIR)
print("Chunk size:", f"{CHUNK_SIZE:,}")
print("Start timestamp:", START_TS)
print("End timestamp (exclusive):", END_TS)

assert START_TS == 1511539200
assert END_TS == 1512316800

# ------------------------------------------------------------
# 2. INITIALIZE STATISTICS
# ------------------------------------------------------------

stats = {
    "total_raw": 0,
    "missing_values": 0,
    "invalid_numeric": 0,
    "invalid_id": 0,
    "invalid_behavior": 0,
    "invalid_timestamp": 0,
    "outside_time_window": 0,
    "total_clean": 0,
    "chunks_processed": 0,
    "output_files": 0
}

behavior_counts = {
    "pv": 0,
    "fav": 0,
    "cart": 0,
    "buy": 0
}

min_clean_ts = None
max_clean_ts = None

start_time = time.time()

# ------------------------------------------------------------
# 3. STREAMING DATA CLEANING
# ------------------------------------------------------------

reader = pd.read_csv(
    CSV_PATH,
    header=None,
    names=COLUMNS,
    dtype="string",
    chunksize=CHUNK_SIZE
)

for chunk_id, chunk in enumerate(reader, start=1):

    stats["chunks_processed"] += 1
    stats["total_raw"] += len(chunk)

    # STEP 1: Remove missing values
    missing_mask = chunk[COLUMNS].isna().any(axis=1)

    stats["missing_values"] += int(missing_mask.sum())
    chunk = chunk.loc[~missing_mask].copy()

    # STEP 2: Normalize numeric columns
    numeric_columns = [
        "user_id",
        "item_id",
        "category_id",
        "timestamp"
    ]

    for col in numeric_columns:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

    invalid_numeric_mask = (
        chunk[numeric_columns].isna().any(axis=1)
    )

    stats["invalid_numeric"] += int(
        invalid_numeric_mask.sum()
    )

    chunk = chunk.loc[~invalid_numeric_mask].copy()

    # STEP 3: Validate integer IDs
    valid_id_mask = (
        (chunk["user_id"] > 0)
        & (chunk["item_id"] > 0)
        & (chunk["category_id"] > 0)
        & (chunk["user_id"] % 1 == 0)
        & (chunk["item_id"] % 1 == 0)
        & (chunk["category_id"] % 1 == 0)
    )

    stats["invalid_id"] += int(
        (~valid_id_mask).sum()
    )

    chunk = chunk.loc[valid_id_mask].copy()

    # STEP 4: Normalize behavior labels
    chunk["behavior"] = (
        chunk["behavior"]
        .str.strip()
        .str.lower()
    )

    valid_behavior_mask = (
        chunk["behavior"].isin(VALID_BEHAVIORS)
    )

    stats["invalid_behavior"] += int(
        (~valid_behavior_mask).sum()
    )

    chunk = chunk.loc[valid_behavior_mask].copy()

    # STEP 5: Validate timestamp integer format
    valid_timestamp_mask = (
        (chunk["timestamp"] % 1 == 0)
    )

    stats["invalid_timestamp"] += int(
        (~valid_timestamp_mask).sum()
    )

    chunk = chunk.loc[valid_timestamp_mask].copy()

    # STEP 6: Temporal window filtering
    valid_time_mask = (
        (chunk["timestamp"] >= START_TS)
        & (chunk["timestamp"] < END_TS)
    )

    stats["outside_time_window"] += int(
        (~valid_time_mask).sum()
    )

    chunk = chunk.loc[valid_time_mask].copy()

    if chunk.empty:
        print(
            f"Chunk {chunk_id}: "
            "No valid interactions"
        )
        continue

    # STEP 7: Convert data types
    for col in numeric_columns:
        chunk[col] = chunk[col].astype("int64")

    # STEP 8: Encode behavior
    chunk["behavior_id"] = (
        chunk["behavior"]
        .map(BEHAVIOR_ENCODING)
        .astype("int8")
    )

    # STEP 9: Sort events within the chunk
    # Global sorting will be done before TGN training.
    chunk = chunk.sort_values(
        ["timestamp", "user_id", "item_id"],
        kind="stable"
    )

    # STEP 10: Update statistics
    stats["total_clean"] += len(chunk)

    counts = chunk["behavior"].value_counts()

    for behavior in behavior_counts:
        behavior_counts[behavior] += int(
            counts.get(behavior, 0)
        )

    current_min = int(chunk["timestamp"].min())
    current_max = int(chunk["timestamp"].max())

    min_clean_ts = (
        current_min
        if min_clean_ts is None
        else min(min_clean_ts, current_min)
    )

    max_clean_ts = (
        current_max
        if max_clean_ts is None
        else max(max_clean_ts, current_max)
    )

    # STEP 11: Save Parquet
    output_file = (
        OUTPUT_DIR / f"part_{chunk_id:04d}.parquet"
    )

    table = pa.Table.from_pandas(
        chunk.reset_index(drop=True),
        preserve_index=False
    )

    pq.write_table(
        table,
        output_file,
        compression="snappy"
    )

    stats["output_files"] += 1

    elapsed = time.time() - start_time

    print(
        f"Chunk {chunk_id:03d} | "
        f"Raw: {stats['total_raw']:,} | "
        f"Clean: {stats['total_clean']:,} | "
        f"Time: {elapsed / 60:.1f} min"
    )

    del chunk, table
    gc.collect()

# ------------------------------------------------------------
# 4. FINAL VALIDATION
# ------------------------------------------------------------

total_removed = (
    stats["missing_values"]
    + stats["invalid_numeric"]
    + stats["invalid_id"]
    + stats["invalid_behavior"]
    + stats["invalid_timestamp"]
    + stats["outside_time_window"]
)

assert (
    stats["total_raw"]
    == stats["total_clean"] + total_removed
), "ERROR: Row counts do not match!"

assert (
    sum(behavior_counts.values())
    == stats["total_clean"]
), "ERROR: Behavior counts do not match!"

# ------------------------------------------------------------
# 5. EXPORT REPORT
# ------------------------------------------------------------

stats["total_removed"] = total_removed

stats["retention_rate_percent"] = round(
    100 * stats["total_clean"] / stats["total_raw"],
    6
)

stats["processing_minutes"] = round(
    (time.time() - start_time) / 60,
    2
)

stats["min_clean_timestamp"] = min_clean_ts
stats["max_clean_timestamp"] = max_clean_ts

stats["behavior_counts"] = behavior_counts

report_path = OUTPUT_DIR / "cleaning_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        stats,
        f,
        ensure_ascii=False,
        indent=4
    )

summary = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in stats.items()
    if not isinstance(value, dict)
])

summary.to_csv(
    OUTPUT_DIR / "cleaning_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 6. PRINT RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL CLEANING RESULTS")
print("=" * 60)

print(f"Raw interactions:      {stats['total_raw']:,}")
print(f"Missing values:        {stats['missing_values']:,}")
print(f"Invalid numeric:       {stats['invalid_numeric']:,}")
print(f"Invalid IDs:           {stats['invalid_id']:,}")
print(f"Invalid behaviors:     {stats['invalid_behavior']:,}")
print(f"Invalid timestamps:    {stats['invalid_timestamp']:,}")
print(f"Outside time window:   {stats['outside_time_window']:,}")
print(f"Total removed:         {stats['total_removed']:,}")
print(f"Clean interactions:    {stats['total_clean']:,}")
print(f"Retention rate:        {stats['retention_rate_percent']}%")

print("\nBehavior distribution:")
for behavior, count in behavior_counts.items():
    print(f"  {behavior:5s}: {count:,}")

if min_clean_ts is not None:
    print("\nClean time range (UTC+8):")

    print(
        pd.to_datetime(
            min_clean_ts,
            unit="s",
            utc=True
        ).tz_convert("Asia/Shanghai")
    )

    print(
        pd.to_datetime(
            max_clean_ts,
            unit="s",
            utc=True
        ).tz_convert("Asia/Shanghai")
    )

print("\nOutput Parquet files:", stats["output_files"])
print("Processing time:", stats["processing_minutes"], "minutes")
print("Report:", report_path)
print("\nCELL 5 COMPLETED SUCCESSFULLY")